# 🌾 FallahTech RAG — Scoring Investissement Série A

**Pipeline RAG complet** avec Groq LLM (GPT-OSS-120B) pour l'analyse du dossier FallahTech SARL.

| Composant | Détail |
|---|---|
| **Embedding** | all-MiniLM-L6-v2 (384-dim) |
| **Vector DB** | ChromaDB (cosine, HNSW) |
| **LLM Principal** | GPT-OSS-120B via Groq |
| **Fallbacks** | GPT-OSS-20B → Compound-Mini |
| **Chunking** | 1000 chars / 200 overlap |

> ⚠️ **Mise à jour modèles (2026)** : `llama-3.3-70b-versatile`, `llama-3.1-8b-instant` et
> `qwen/qwen3-32b` ont été décommissionnés par Groq (erreur `404 model_not_found`).
> Le notebook utilise désormais les modèles de production actuels et vérifie
> dynamiquement leur disponibilité via l'API `/v1/models`.

---

## ⚙️ Cellule 1 — Installation des dépendances

In [ ]:
!pip install -q chromadb sentence-transformers groq pypdf openpyxl plotly ipywidgets
print("✅ Toutes les dépendances installées.")

## 🔑 Cellule 2 — Clé API Groq

Entrez votre clé API Groq (gratuite sur [console.groq.com](https://console.groq.com)).

In [ ]:
import os
from google.colab import userdata

# Méthode 1: Via Colab Secrets (recommandé)
try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("✅ Clé API chargée depuis Colab Secrets.")
except Exception:
    pass

# Méthode 2: Saisie manuelle (si Secrets non configuré)
if not os.environ.get("GROQ_API_KEY"):
    import getpass
    os.environ["GROQ_API_KEY"] = getpass.getpass("🔑 Entrez votre GROQ_API_KEY: ")
    print("✅ Clé API configurée manuellement.")

## 📁 Cellule 3 — Upload des documents FallahTech

Uploadez les **6 fichiers** de la Data Room (5 PDFs + 1 Excel).

In [ ]:
import os
from google.colab import files

DATA_DIR = "/content/fallahtech_data"
os.makedirs(DATA_DIR, exist_ok=True)

print("📤 Uploadez les fichiers de la Data Room FallahTech :")
print("   - 1.1_Statuts_FallahTech_*.pdf")
print("   - 1.2_Contrat_Cooperative_*.pdf")
print("   - 2.1_Etats_Financiers_*.pdf")
print("   - 3.1_Registre_Personnel_*.pdf")
print("   - 4.1_Etude_Marche_*.pdf")
print("   - FallahTech_BusinessPlan_*.xlsx")
print()

uploaded = files.upload()

for filename in uploaded:
    dest = os.path.join(DATA_DIR, filename)
    with open(dest, "wb") as f:
        f.write(uploaded[filename])
    print(f"  ✅ {filename}")

print(f"\n📂 {len(uploaded)} fichiers uploadés dans {DATA_DIR}")

## 📥 Cellule 4 — Ingestion + Chunking + Embeddings

Pipeline complet : PDF/Excel → raw_docs.json → chunks → ChromaDB.

In [ ]:
import os, json, re, time
from pypdf import PdfReader
import openpyxl
from sentence_transformers import SentenceTransformer
import chromadb

DATA_DIR = "/content/fallahtech_data"
CHROMA_DIR = "/content/chroma_db"
RAW_DOCS_FILE = "/content/raw_docs.json"
COLLECTION_NAME = "fallahtech_docs"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

# ========== INGESTION ==========
def extract_pdf_text(filepath):
    reader = PdfReader(filepath)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        text = text.replace("\x00", "")
        if text.strip():
            pages.append({"page": i + 1, "text": text.strip()})
    return pages

def extract_xlsx_text(filepath):
    wb = openpyxl.load_workbook(filepath, data_only=True)
    sheets = []
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        rows = []
        for row in ws.iter_rows(values_only=True):
            row_text = " | ".join([str(c) if c is not None else "" for c in row])
            if row_text.strip().replace("|", "").strip():
                rows.append(row_text)
        if rows:
            sheets.append({"sheet": sheet_name, "text": "\n".join(rows)})
    return sheets

print("📥 Ingestion des documents...")
documents = []
for filename in sorted(os.listdir(DATA_DIR)):
    filepath = os.path.join(DATA_DIR, filename)
    source_name = filename.split("_1775661")[0] if "_1775661" in filename else filename.rsplit(".", 1)[0]
    if filename.endswith(".pdf"):
        pages = extract_pdf_text(filepath)
        for p in pages:
            documents.append({"source": source_name + ".pdf", "page": p["page"], "text": p["text"]})
        print(f"  ✅ {filename} → {len(pages)} pages")
    elif filename.endswith(".xlsx"):
        sheets = extract_xlsx_text(filepath)
        for s in sheets:
            documents.append({"source": source_name + ".xlsx", "sheet": s["sheet"], "text": s["text"]})
        print(f"  ✅ {filename} → {len(sheets)} feuilles")

with open(RAW_DOCS_FILE, "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)
print(f"\n📄 {len(documents)} segments extraits → raw_docs.json")

# ========== CHUNKING + EMBEDDINGS ==========
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk.strip())
        start += chunk_size - overlap
    return chunks

print("\n🧠 Chargement du modèle d'embedding all-MiniLM-L6-v2...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

all_chunks, all_metas, all_ids = [], [], []
idx = 0
for doc in documents:
    chunks = chunk_text(doc["text"])
    source = doc.get("source", "unknown")
    page = doc.get("page", doc.get("sheet", "N/A"))
    for c in chunks:
        all_chunks.append(c)
        all_metas.append({"source": source, "page": str(page)})
        all_ids.append(f"chunk_{idx}")
        idx += 1

print(f"✂️  {len(all_chunks)} chunks créés (taille={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")

print("💾 Vectorisation et stockage dans ChromaDB...")
client = chromadb.PersistentClient(path=CHROMA_DIR)
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
collection = client.create_collection(name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"})

embeddings = embed_model.encode(all_chunks, show_progress_bar=True).tolist()
collection.add(documents=all_chunks, embeddings=embeddings, metadatas=all_metas, ids=all_ids)

print(f"\n✅ Pipeline terminé ! {len(all_chunks)} chunks indexés dans ChromaDB.")
print(f"   Collection: {COLLECTION_NAME} | Métrique: cosine | Index: HNSW")

## 🧠 Cellule 5 — Moteur RAG (Retrieval + LLM)

Configuration du pipeline RAG complet : retrieval sémantique + Groq LLM avec fallback.

**🔧 Corrections appliquées :**
1. `MODELS` mis à jour vers les modèles Groq **actifs** (les anciens renvoyaient `404 model_not_found`).
2. Filtrage dynamique via `GET /openai/v1/models` — seuls les modèles réellement
   disponibles sur votre clé sont conservés.
3. `call_llm` journalise le modèle en échec avant de basculer sur le fallback.
4. `reasoning_effort="low"` pour les modèles GPT-OSS (évite les traces de raisonnement
   qui cassent le format `===== RÉPONSE =====`).

In [ ]:
import os, re, time, json
import requests
from groq import Groq
from difflib import SequenceMatcher
from sentence_transformers import SentenceTransformer
import chromadb
import plotly.graph_objects as go
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# ========== CONFIGURATION ==========
CHROMA_DIR = "/content/chroma_db"
COLLECTION_NAME = "fallahtech_docs"

# ✅ CORRECTIF : anciens modèles décommissionnés par Groq
#   llama-3.3-70b-versatile / llama-3.1-8b-instant / qwen/qwen3-32b  -> 404 model_not_found
MODELS = ["openai/gpt-oss-120b", "openai/gpt-oss-20b", "groq/compound-mini"]
K_VALUE = 5  # Nombre de chunks par requête

# ---------- Vérification dynamique de la disponibilité des modèles ----------
def available_models():
    """Retourne la liste des model_id réellement servis par la clé API courante."""
    try:
        r = requests.get(
            "https://api.groq.com/openai/v1/models",
            headers={"Authorization": f"Bearer {os.environ.get('GROQ_API_KEY','')}"},
            timeout=20,
        )
        r.raise_for_status()
        return sorted(m["id"] for m in r.json().get("data", []))
    except Exception as e:
        print(f"⚠️  Impossible de lister les modèles ({e}) — liste par défaut conservée.")
        return None

_avail = available_models()
if _avail:
    _kept = [m for m in MODELS if m in _avail]
    if not _kept:
        # Repli intelligent : on prend les modèles texte les plus capables disponibles
        _kept = [m for m in ("openai/gpt-oss-120b", "openai/gpt-oss-20b",
                             "qwen/qwen3.6-27b", "groq/compound-mini") if m in _avail]
    MODELS = _kept or MODELS
    print(f"✅ Modèles actifs retenus : {MODELS}")

# Les modèles GPT-OSS sont des modèles de raisonnement -> on limite l'effort
REASONING_MODELS = ("openai/gpt-oss-120b", "openai/gpt-oss-20b", "openai/gpt-oss-safeguard-20b")

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = chroma_client.get_collection(COLLECTION_NAME)

SYSTEM_PROMPT = """Tu es un analyste financier senior d'un fonds d'investissement franco-tunisien. Tu instruis le dossier Série A de FallahTech SARL — startup AgriTech tunisienne basée à Sousse.
RÈGLES ABSOLUES :
(1) JAMAIS inventer de données. Cite UNIQUEMENT les documents fournis.
(2) Si une info n'est PAS dans les documents, écris « Non disponible dans le corpus ». N'invente JAMAIS de biographies, années d'expérience, diplômes ou qualifications.
(3) Cite TOUJOURS [SOURCE: nom_fichier] après chaque fait.
(4) Chiffres EXACTS en TND, tels que fournis dans le Référentiel Vérifié.
(5) FallahTech = AgriTech, PAS télécommunications.
(6) Réponds en français professionnel.
(7) NE répète JAMAIS le même paragraphe ou la même idée deux fois. Sois concis et structuré.
(8) Respecte STRICTEMENT le format demandé. Commence directement par le format obligatoire."""

CORPUS_REFERENCE = """=== RÉFÉRENTIEL VÉRIFIÉ — FALLAHTECH SARL ===
[SOURCE: Etats_Financiers_Historiques + BusinessPlan_Complet + Registre_Personnel + Etude_Marche + Statuts]

━━━ DONNÉES FINANCIÈRES ━━━
CHIFFRE D'AFFAIRES (TND): 2023: 250 000 | 2024: 780 000 (+212%) | 2025: 1 650 000 (+111.5%)
Prévisions: 2026: 2 520 000 | 2027: 3 657 000 | 2028: 5 118 000 | 2029: 6 910 000
RÉSULTAT NET (TND): 2023: -50 000 | 2024: -40 000 | 2025: 45 000 (1ère rentabilité) | 2026: 321 000 | 2027: 701 000
EBITDA (TND): 2025: 180 000 | 2026: 420 000 | 2027: 810 000 | 2028: 1 330 000 | 2029: 1 820 000
MARGES: Brute: 2023: 60%, 2024: 64%, 2025: 70% | EBITDA: 2025: 10.9%, 2026: 16.7%, 2027: 22.1%
TRÉSORERIE: 2023: 100 000 | 2024: 220 000 | 2025: 510 000 TND
RATIOS: Ratio courant: 2024: 9.43 | 2025: 1.56 | ROE 2025: 9.0% | ROA 2025: 4.8%
DETTES FOURNISSEURS: 2025: 300 000 TND (x15 vs 2024 - Alerte)
RISQUES: Dépendance marché AgriTech | Concentration client (5 coopératives=60% CA) | Ratio courant 9.43→1.56

━━━ MÉTRIQUES SaaS ━━━
Utilisateurs: 2025: 3 500 | 2026: 5 250 | 2027: 7 613 | ARPU: 470 TND/an
CAC: 500 TND | LTV: 2 640 TND | LTV/CAC: 5.28 | Churn: 1.5%/mois | Rétention: 82%

━━━ ÉQUIPE — 18 EMPLOYÉS ━━━
[SOURCE: Registre_Personnel]
CEO: Sami BEN YOUSSEF (72k TND/an) | CTO: Amira TRABELSI (66k TND/an)
Répartition: 7 Tech, 5 Sales, 4 Agro, 1 Direction, 1 Ops | Masse salariale: 630 000 TND

━━━ MARCHÉ AGRITECH MAGHREB ━━━
[SOURCE: Etude_Marche_Synthese]
TAM: 516 000 exploitations | SAM: 120 000 | Part: ~3% | Prix: 15-30 TND/mois vs >200 (importées)
Expansion: Algérie 1.5M exploitations | Maroc 1.5M, Plan Génération Green

━━━ STRUCTURE JURIDIQUE ━━━
SARL | Siège: Novation City, Sousse | Capital: 500 000 TND | Valorisation pre-money: 12M TND
Actionnaires: CEO 40%, CTO 35%, Seed Fund AgriVentures TN 25%"""

SCORING_CRITERIA = {
    "Santé Financière": {
        "weight": 0.40, "icon": "💰",
        "query": "Analyse financière FallahTech : chiffre d'affaires, résultat net, EBITDA, marge brute, trésorerie, ratio courant, solvabilité, évolution 2023-2025, rentabilité, dettes fournisseurs",
        "description": "Solidité financière, rentabilité, liquidité et solvabilité sur 3 exercices"
    },
    "Traction Commerciale": {
        "weight": 0.30, "icon": "📈",
        "query": "Traction commerciale FallahTech : croissance chiffre d'affaires, nombre abonnés, taux de rétention, coopératives partenaires, expansion géographique, gouvernorats couverts",
        "description": "Croissance du CA, base clients, rétention et dynamique commerciale"
    },
    "Qualité de l'Équipe": {
        "weight": 0.15, "icon": "👥",
        "query": "Équipe FallahTech : effectif, organigramme, compétences clés, CEO CTO, départements, salaires, recrutements, profils techniques et agronomiques",
        "description": "Compétences, structure, expérience et capacité d'exécution"
    },
    "Opportunité de Marché": {
        "weight": 0.15, "icon": "🌍",
        "query": "Marché AgriTech FallahTech : TAM SAM SOM, concurrence, positionnement prix, expansion Maghreb, avantage compétitif",
        "description": "Taille du marché, positionnement concurrentiel et potentiel de croissance"
    }
}

# ========== FONCTIONS RAG ==========
def query_documents(query_text, n_results=5):
    query_emb = embed_model.encode([query_text]).tolist()
    return collection.query(query_embeddings=query_emb, n_results=n_results,
                            include=["documents", "metadatas", "distances"])

def _reformat_pipe_table(text):
    lines = text.split("\n")
    headers = None
    output = []
    for line in lines:
        if "|" not in line:
            output.append(line); continue
        cells = [c.strip() for c in line.split("|")]
        while cells and not cells[-1]: cells.pop()
        if not cells or len(cells) < 2:
            output.append(line); continue
        potential_years = [c for c in cells[1:] if re.match(r"^20[2-3]\d(\s*\(.*\))?$", c.strip())]
        if len(potential_years) >= 2:
            headers = cells
            output.append(f"[COLONNES: {' | '.join(cells)}]")
            continue
        if headers and len(cells) >= 2:
            label = cells[0]
            values = [c for c in cells[1:] if c.strip()]
            if not values:
                output.append(f"\n--- {label} ---"); continue
            for j, val in enumerate(cells[1:], 1):
                if j < len(headers) and val:
                    year = headers[j]
                    if year and val not in ("", " "):
                        output.append(f"  {label} en {year}: {val}")
        else:
            output.append(line)
    return "\n".join(output)

def retrieve_context(query, n_results=5):
    results = query_documents(query, n_results=n_results)
    priority_contexts = [CORPUS_REFERENCE]
    sources = {"Référentiel Vérifié (tous documents)"}
    semantic_contexts, seen_chunks = [], set()
    if results and results["documents"] and results["documents"][0]:
        for i, doc in enumerate(results["documents"][0]):
            meta = results["metadatas"][0][i] if results["metadatas"] else {}
            source = meta.get("source", "unknown")
            page = meta.get("page", "N/A")
            sources.add(f"{source} (p.{page})")
            chunk_key = doc[:100]
            if chunk_key not in seen_chunks:
                seen_chunks.add(chunk_key)
                semantic_contexts.append(doc)
    all_contexts = priority_contexts + semantic_contexts
    reformatted = [_reformat_pipe_table(c) if "|" in c else c for c in all_contexts]
    MAX_CONTEXT_CHARS = 5500
    final, total = [], 0
    for ctx in reformatted:
        if total + len(ctx) > MAX_CONTEXT_CHARS and final: break
        final.append(ctx); total += len(ctx) + 10
    return "\n\n---\n\n".join(final), list(sources)

def call_llm(system_prompt, user_prompt, model_index=0):
    """Appel Groq avec cascade de fallback + log explicite du modèle en échec."""
    client = Groq(api_key=os.environ.get("GROQ_API_KEY", ""))
    model = MODELS[model_index] if model_index < len(MODELS) else MODELS[0]
    kwargs = dict(
        model=model,
        messages=[{"role": "system", "content": system_prompt},
                  {"role": "user", "content": user_prompt}],
        temperature=0.4, max_tokens=1500, top_p=0.9,
    )
    # ✅ évite les traces de raisonnement qui cassent le format de sortie
    if model in REASONING_MODELS:
        kwargs["reasoning_effort"] = "low"
    try:
        response = client.chat.completions.create(**kwargs)
        content = response.choices[0].message.content or "Réponse vide."
        return content, model
    except Exception as e:
        print(f"⚠️  Échec du modèle '{model}' : {type(e).__name__} — {e}")
        if model_index < len(MODELS) - 1:
            print(f"   ↪️  Bascule vers le fallback '{MODELS[model_index + 1]}'...")
            time.sleep(3)
            return call_llm(system_prompt, user_prompt, model_index + 1)
        return f"Erreur LLM ({model}): {e}", model

def _similarity(a, b):
    return SequenceMatcher(None, a.strip(), b.strip()).ratio()

def clean_scoring_response(text):
    if not text or text.startswith("Erreur"): return text
    # Remove stray ** artifacts from LLM output
    text = re.sub(r"\*\*$", "", text)
    text = re.sub(r"(?m)^\*\*\s*", "", text)
    text = re.sub(r"\*\*", "", text)  # Remove ALL remaining **
    # Remove standalone ANALYSE / CITATIONS / SOURCES pseudo-headers
    text = re.sub(r"(?m)^(ANALYSE|CITATIONS|SOURCES)\s*$", "", text)
    # Strip SCORE trailing **
    text = re.sub(r"(SCORE\s*:\s*\d+(?:[.,]\d+)?\s*/\s*10)\s*\*\*", r"\1", text)
    # Keep only from SCORE: onwards
    score_match = re.search(r"SCORE\s*:?\s*\d+(?:[.,]\d+)?\s*/\s*10", text)
    if score_match: text = text[score_match.start():].strip()
    # Remove duplicate ANALYSE blocks
    analyse_blocks = list(re.finditer(r"ANALYSE\s*:?\s*(?:\n|.)", text))
    if len(analyse_blocks) >= 2:
        text = text[:analyse_blocks[1].start()].strip()
    # Paragraph-level deduplication
    paragraphs = re.split(r"\n{2,}", text)
    seen, deduped = [], []
    for para in paragraphs:
        pc = re.sub(r"\s+", " ", para).strip()
        if not pc: continue
        if not any(_similarity(pc, p) > 0.5 for p in seen):
            seen.append(pc); deduped.append(para)
    return "\n\n".join(deduped)

def clean_qa_response(text):
    if not text or text.startswith("Erreur"): return text
    text = re.sub(r"(?m)^\*\*\s*", "", text)
    if "===== SOURCES =====" in text:
        text = re.sub(r"\n\*\*[Ss]ources?\*\*.*?(?=\n===== |$)", "", text, flags=re.DOTALL)
    paragraphs = re.split(r"\n{2,}", text)
    seen, deduped = [], []
    for para in paragraphs:
        pc = re.sub(r"\s+", " ", para).strip()
        if not pc: continue
        if not any(_similarity(pc, p) > 0.75 for p in seen):
            seen.append(pc); deduped.append(para)
    return "\n\n".join(deduped)

def parse_score(text):
    for pattern in [r"SCORE\s*:\s*(\d+(?:[.,]\d+)?)\s*/\s*10",
                    r"SCORE\s*:\s*(\d+(?:[.,]\d+)?)",
                    r"(\d+(?:[.,]\d+)?)\s*/\s*10",
                    r"[Ss]core\s*(?:global|final|:)?\s*:?\s*(\d+(?:[.,]\d+)?)"]:
        m = re.search(pattern, text)
        if m:
            val = float(m.group(1).replace(",", "."))
            if 0 <= val <= 10: return val
    return 5.0

print("✅ Moteur RAG prêt ! (Retrieval + LLM + Charts)")
print(f"   Modèle principal: {MODELS[0]}")
arrow = ' → '
print(f"   Fallbacks: {arrow.join(MODELS[1:]) if len(MODELS) > 1 else 'aucun'}")
print(f"   Collection ChromaDB: {COLLECTION_NAME} ({collection.count()} chunks)")

---

## ❓ Cellule 6 — Question Libre (Q&A)

Interface interactive pour poser des questions sur le dossier FallahTech. Le RAG recherche les passages pertinents et génère une réponse traçable.

In [ ]:
#@title ❓ Assistant Q&A — Posez votre question sur FallahTech
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── UI Widgets ──
question_input = widgets.Textarea(
    placeholder="Ex: Quel est le chiffre d'affaires de FallahTech en 2025 et quels sont les risques financiers ?",
    layout=widgets.Layout(width="100%", height="80px"),
    style={"description_width": "0px"}
)
k_slider = widgets.IntSlider(value=5, min=3, max=15, step=1, description="Chunks (K):")
submit_btn = widgets.Button(description="🔍 Obtenir la Réponse RAG", button_style="primary",
                             layout=widgets.Layout(width="100%", height="40px"))
output_area = widgets.Output()

def build_qa_prompt(question, context, k):
    return f"""QUESTION: {question}

DOCUMENTS ({k} chunks du corpus FallahTech):
{context}

INSTRUCTIONS:
- Analyse TOUS les documents fournis attentivement.
- Cherche les chiffres exacts dans les tableaux.
- CITE les sources avec [SOURCE: nom_fichier].

FORMAT DE RÉPONSE OBLIGATOIRE:
===== RÉPONSE =====
[Réponse structurée et détaillée avec citations [SOURCE: fichier]]

===== SOURCES =====
[Liste numérotée des sources utilisées]

===== CONFIANCE =====
[Élevée/Modérée/Basse — justification]"""

def on_submit(btn):
    question = question_input.value.strip()
    if not question:
        with output_area:
            clear_output()
            display(HTML("<p style='color:#ef5350;'>⚠️ Veuillez saisir une question.</p>"))
        return
    with output_area:
        clear_output()
        display(HTML("<div style='padding:16px;background:#1a237e;border-radius:8px;color:#90caf9;'>🔄 Recherche dans le corpus et génération de la réponse...</div>"))

    k = k_slider.value
    context, sources = retrieve_context(question, n_results=k)
    prompt = build_qa_prompt(question, context, k)
    time.sleep(2)
    response, model_used = call_llm(SYSTEM_PROMPT, prompt)
    cleaned = clean_qa_response(response)

    with output_area:
        clear_output()
        # Header
        display(HTML(f"""
        <div style='padding:8px 16px;background:#263238;border-radius:8px 8px 0 0;margin-top:8px;'>
            <b style='color:#4fc3f7;'>Modèle:</b> <code style='color:#aed581;'>{model_used}</code>
            &nbsp;|&nbsp;
            <b style='color:#4fc3f7;'>Chunks:</b> <span style='color:#fff;'>{k}</span>
        </div>
        """))
        # Response
        formatted = cleaned.replace("\n", "<br>")
        display(HTML(f"""
        <div style='padding:16px;background:#1a1f2e;border-left:4px solid #4fc3f7;border-radius:0 0 8px 8px;color:#e0e0e0;font-size:14px;line-height:1.8;'>
            {formatted}
        </div>
        """))
        # Sources
        if sources:
            src_tags = " ".join([f"<span style='background:#1a237e;color:#90caf9;padding:3px 10px;border-radius:6px;font-size:12px;margin:2px;display:inline-block;'>{s}</span>" for s in sources])
            display(HTML(f"<div style='margin-top:8px;'><b style='color:#4fc3f7;'>Sources consultées:</b><br>{src_tags}</div>"))

submit_btn.on_click(on_submit)

# ── Affichage ──
display(HTML("<h3 style='color:#4fc3f7;'>❓ Assistant Q&A — Corpus FallahTech</h3>"))
display(HTML("<p style='color:#90a4ae;'>Posez une question analytique sur le dossier FallahTech. Le système RAG recherchera les passages pertinents.</p>"))
display(question_input)
display(k_slider)
display(submit_btn)
display(output_area)

display(HTML("""
<div style='margin-top:16px;padding:12px;background:#263238;border-radius:8px;'>
<b style='color:#4fc3f7;'>Exemples de questions :</b>
<ul style='color:#b0bec5;'>
<li>Quel est le chiffre d'affaires de FallahTech en 2025 et sa croissance par rapport à 2024 ?</li>
<li>Quels sont les principaux risques financiers identifiés dans le dossier ?</li>
<li>Quelle est la structure de l'équipe FallahTech et les compétences clés ?</li>
<li>Quel est le positionnement prix de FallahTech par rapport à la concurrence ?</li>
<li>La valorisation pre-money de 12M TND est-elle justifiée ?</li>
</ul></div>
"""))

---

## 📊 Cellule 7 — Scoring Investissement Complet

Lance le scoring multicritère (4 critères pondérés) avec charts professionnels Plotly.

In [ ]:
#@title 📊 Lancer le Scoring Investissement Complet
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML, clear_output
import time

K_SCORING = 8  #@param {type:"slider", min:3, max:15, step:1}

def build_scoring_prompt(criteria_name, context, k):
    if k <= 5: length = "Sois concis (~100 mots)."
    elif k <= 10: length = "Style journalistique (~200 mots)."
    else: length = "Analyse approfondie (~300 mots)."
    return f"""CRITÈRE À ÉVALUER : {criteria_name}

CONTEXTE FOURNI :
{context}

INSTRUCTIONS STRICTES :
{length}
- Utilise les données chiffrées exactes du Référentiel Vérifié.
- Cite les sources avec [SOURCE: nom_document].
- NE répète JAMAIS. Commence directement par le format obligatoire.

FORMAT OBLIGATOIRE :
SCORE: [0.0-10.0] / 10
ANALYSE: [texte analytique structuré avec chiffres et citations]"""

def create_gauge(score, title):
    color = "#66BB6A" if score >= 7 else "#FFA726" if score >= 5 else "#EF5350"
    fig = go.Figure(go.Indicator(
        mode="gauge+number", value=score,
        title={"text": title, "font": {"size": 14, "color": "#FAFAFA"}},
        number={"suffix": "/10", "font": {"size": 24, "color": "#FAFAFA"}},
        gauge={"axis": {"range": [0, 10], "tickcolor": "#546E7A", "tickfont": {"color": "#90A4AE"}},
               "bar": {"color": color, "thickness": 0.7},
               "bgcolor": "#1A1F2E", "bordercolor": "#2A3142",
               "steps": [{"range": [0, 3.5], "color": "rgba(239,83,80,0.15)"},
                         {"range": [3.5, 6.5], "color": "rgba(255,167,38,0.15)"},
                         {"range": [6.5, 10], "color": "rgba(102,187,106,0.15)"}]}
    ))
    fig.update_layout(height=200, margin=dict(l=20, r=20, t=50, b=10),
                      paper_bgcolor="rgba(0,0,0,0)", font={"color": "#FAFAFA"})
    return fig

def create_bar_chart(scores_dict):
    names = list(scores_dict.keys())
    scores = list(scores_dict.values())
    colors = ["#66BB6A" if s >= 7 else "#FFA726" if s >= 5 else "#EF5350" for s in scores]
    fig = go.Figure(go.Bar(x=names, y=scores, marker_color=colors,
                           text=[f"{s:.1f}/10" for s in scores], textposition="outside",
                           textfont={"color": "#FAFAFA", "size": 14}))
    fig.update_layout(title={"text": "Scores par Critère", "font": {"size": 18, "color": "#FAFAFA"}},
                      yaxis={"range": [0, 11], "gridcolor": "#2A3142", "color": "#90A4AE"},
                      xaxis={"color": "#90A4AE"}, height=380,
                      paper_bgcolor="rgba(26,31,46,0.95)", plot_bgcolor="rgba(0,0,0,0)",
                      font={"color": "#FAFAFA"}, margin=dict(l=40, r=40, t=60, b=80))
    return fig

def create_radar_chart(scores_dict):
    names = list(scores_dict.keys()) + [list(scores_dict.keys())[0]]
    scores = list(scores_dict.values()) + [list(scores_dict.values())[0]]
    fig = go.Figure(go.Scatterpolar(r=scores, theta=names, fill="toself",
                                     fillcolor="rgba(79,195,247,0.2)",
                                     line={"color": "#4FC3F7", "width": 2},
                                     marker={"size": 8, "color": "#4FC3F7"}))
    fig.update_layout(
        polar={"bgcolor": "rgba(0,0,0,0)",
               "radialaxis": {"visible": True, "range": [0, 10], "gridcolor": "#2A3142", "color": "#90A4AE"},
               "angularaxis": {"color": "#90A4AE"}},
        title={"text": "Profil Multicritère", "font": {"size": 18, "color": "#FAFAFA"}},
        height=420, paper_bgcolor="rgba(26,31,46,0.95)", plot_bgcolor="rgba(0,0,0,0)",
        font={"color": "#FAFAFA"}, margin=dict(l=60, r=60, t=60, b=40))
    return fig

# ========== SCORING PIPELINE ==========
display(HTML("<h2 style='color:#4fc3f7;text-align:center;'>📊 Scoring Investissement FallahTech — Série A</h2>"))
display(HTML(f"<p style='text-align:center;color:#90a4ae;'>Grille : Santé Financière (40%) | Traction Commerciale (30%) | Équipe (15%) | Marché (15%) | K = {K_SCORING}</p>"))

scores, analyses, all_sources, models_used = {}, {}, {}, {}
criteria_list = list(SCORING_CRITERIA.items())

for i, (crit_name, crit_info) in enumerate(criteria_list):
    print(f"\n{'='*60}")
    print(f"  {crit_info['icon']}  Analyse en cours : {crit_name} ({i+1}/{len(criteria_list)})")
    print(f"{'='*60}")

    context, src = retrieve_context(crit_info["query"], n_results=K_SCORING)
    prompt = build_scoring_prompt(crit_name, context, K_SCORING)
    time.sleep(3)  # Rate limiting Groq
    response, model = call_llm(SYSTEM_PROMPT, prompt)
    score = parse_score(response)

    scores[crit_name] = score
    analyses[crit_name] = response
    all_sources[crit_name] = src
    models_used[crit_name] = model

    print(f"  ✅ Score: {score:.1f}/10 | Modèle: {model}")

# ========== SCORE GLOBAL ==========
global_score = sum(scores[c] * SCORING_CRITERIA[c]["weight"] for c in scores)
if global_score >= 7.0:
    rec, rec_color = "✅ INVESTIR", "#66BB6A"
elif global_score >= 5.0:
    rec, rec_color = "⚠️ INVESTIR SOUS CONDITIONS", "#FFA726"
else:
    rec, rec_color = "❌ NO-GO", "#EF5350"

sep = '═'*60
print(f"\n{sep}")
print(f"  SCORE GLOBAL PONDÉRÉ : {global_score:.1f}/10")
print(f"  RECOMMANDATION : {rec}")
print(f"{sep}")

# ========== RECOMMENDATION BOX ==========
display(HTML(f"""
<div style='background:linear-gradient(135deg,#1a237e,#283593);border:2px solid #3f51b5;
            border-radius:12px;padding:24px;text-align:center;margin:16px 0;'>
    <div style='font-size:14px;color:#90a4ae;'>RECOMMANDATION FINALE</div>
    <div style='font-size:32px;font-weight:800;color:{rec_color};margin:8px 0;'>{rec}</div>
    <div style='font-size:18px;color:#fafafa;'>Score Global Pondéré : <b style='color:{rec_color};'>{global_score:.1f}/10</b></div>
</div>
"""))

# ========== CHARTS ==========
print("\n📈 Génération des graphiques professionnels...")

# Bar Chart
fig_bar = create_bar_chart(scores)
fig_bar.show()

# Radar Chart
fig_radar = create_radar_chart(scores)
fig_radar.show()

# Gauge Charts
for crit_name, score in scores.items():
    icon = SCORING_CRITERIA[crit_name]["icon"]
    fig_gauge = create_gauge(score, f"{icon} {crit_name}")
    fig_gauge.show()

# ========== DETAILED ANALYSES ==========
display(HTML("<h3 style='color:#4fc3f7;margin-top:24px;'>📝 Analyses Détaillées par Critère</h3>"))

for crit_name, analysis in analyses.items():
    info = SCORING_CRITERIA[crit_name]
    score = scores[crit_name]
    model = models_used[crit_name]
    src = all_sources[crit_name]
    cleaned = clean_scoring_response(analysis)
    formatted = cleaned.replace("\n", "<br>")
    src_tags = " ".join([f"<span style='background:#1a237e;color:#90caf9;padding:2px 8px;border-radius:5px;font-size:11px;margin:2px;display:inline-block;'>{s}</span>" for s in src])

    score_color = "#66BB6A" if score >= 7 else "#FFA726" if score >= 5 else "#EF5350"
    display(HTML(f"""
    <div style='margin:12px 0;border:1px solid #2a3142;border-radius:10px;overflow:hidden;'>
        <div style='padding:12px 16px;background:linear-gradient(135deg,#1a1f2e,#252b3b);'>
            <span style='font-size:18px;'>{info['icon']}</span>
            <b style='color:#4fc3f7;font-size:16px;'> {crit_name}</b>
            <span style='float:right;font-size:20px;font-weight:800;color:{score_color};'>{score:.1f}/10</span>
            <br><span style='color:#78909c;font-size:12px;'>Poids: {int(info['weight']*100)}% | Modèle: {model}</span>
        </div>
        <div style='padding:16px;background:#1a1f2e;border-left:4px solid #4fc3f7;color:#e0e0e0;font-size:13px;line-height:1.7;'>
            {formatted}
        </div>
        <div style='padding:8px 16px;background:#0d1117;'>
            <b style='color:#78909c;font-size:11px;'>Sources:</b> {src_tags}
        </div>
    </div>
    """))

# ========== SUMMARY TABLE ==========
display(HTML("<h3 style='color:#4fc3f7;margin-top:24px;'>📊 Tableau Récapitulatif</h3>"))
table_rows = ""
for c in scores:
    w = SCORING_CRITERIA[c]["weight"]
    s = scores[c]
    sc = "#66BB6A" if s >= 7 else "#FFA726" if s >= 5 else "#EF5350"
    table_rows += f"<tr style='border-bottom:1px solid #2a3142;'><td style='padding:8px;color:#e0e0e0;'>{SCORING_CRITERIA[c]['icon']} {c}</td><td style='text-align:center;color:#90a4ae;'>{int(w*100)}%</td><td style='text-align:center;color:{sc};font-weight:700;'>{s:.1f}/10</td><td style='text-align:center;color:#4fc3f7;'>{s*w:.2f}</td></tr>"

display(HTML(f"""
<table style='width:100%;border-collapse:collapse;background:#1a1f2e;border-radius:8px;overflow:hidden;'>
<thead><tr style='background:#0d47a1;'>
<th style='padding:10px;color:#fff;text-align:left;'>Critère</th>
<th style='padding:10px;color:#fff;text-align:center;'>Poids</th>
<th style='padding:10px;color:#fff;text-align:center;'>Score</th>
<th style='padding:10px;color:#fff;text-align:center;'>Pondéré</th>
</tr></thead>
<tbody>{table_rows}
<tr style='background:#1a237e;font-weight:800;'>
<td style='padding:10px;color:#fff;' colspan='2'>TOTAL</td>
<td></td>
<td style='text-align:center;color:{rec_color};font-size:18px;'>{global_score:.1f}/10</td>
</tr></tbody></table>
"""))

# ========== JSON EXPORT ==========
export_data = {
    "entreprise": "FallahTech SARL",
    "type_analyse": "Scoring Investissement Serie A - T3",
    "score_global": round(global_score, 2),
    "recommandation": rec,
    "scores_detailles": {c: round(s, 2) for c, s in scores.items()},
    "ponderations": {c: SCORING_CRITERIA[c]["weight"] for c in scores},
    "modeles_utilises": models_used,
    "parametres_rag": {"k": K_SCORING, "chunk_size": 1000, "overlap": 200, "embedding": "all-MiniLM-L6-v2"}
}
with open("/content/fallahtech_scoring_report.json", "w", encoding="utf-8") as f:
    json.dump(export_data, f, ensure_ascii=False, indent=2)

print(f"\n✅ Rapport JSON exporté : /content/fallahtech_scoring_report.json")
print(json.dumps(export_data, ensure_ascii=False, indent=2))

---

## 💾 Cellule 8 — Téléchargement du rapport JSON

In [ ]:
from IPython.display import FileLink, display
from google.colab import files

path = "/content/fallahtech_scoring_report.json"

# Affiche un FileLink (Jupyter-style)
display(FileLink(path))

# Téléchargement automatique (Colab)
files.download(path)
print("✅ Rapport téléchargé.")